# LogosGraph Exploration

This notebook demonstrates how to explore the Bible graph database containing:
- ~31,000 verse nodes from the original 73-book Bible (CPDV)
- ~340,000+ cross-reference edges from Treasury of Scripture Knowledge (TSK)

We'll cover basic queries, verse exploration, passage groups, path finding, graph analytics, and visualization.

## Imports

Load the database connection, query modules, and visualization library.

In [1]:
import sys
sys.path.insert(0, '../..')

from src.db.connection import get_connection
from src.queries import sample_queries as sq
from src.queries.analytics import GDSAnalytics

from pyvis.network import Network

## Configuration

Establish connection to Neo4j database.

In [2]:
conn = get_connection()
conn.connect()
conn.verify()
print("Connected to Neo4j!")

Connected to Neo4j!


### Color scheme for books

Define colors for different sections of the Bible to make visualizations more informative.

In [3]:
# Color scheme by book category
BOOK_COLORS = {
    # Pentateuch (Torah) - Blue
    "GEN": "#3498db", "EXO": "#3498db", "LEV": "#3498db", "NUM": "#3498db", "DEU": "#3498db",
    # Historical Books - Green
    "JOS": "#27ae60", "JDG": "#27ae60", "RUT": "#27ae60", "1SA": "#27ae60", "2SA": "#27ae60",
    "1KI": "#27ae60", "2KI": "#27ae60", "1CH": "#27ae60", "2CH": "#27ae60", "EZR": "#27ae60",
    "NEH": "#27ae60", "EST": "#27ae60",
    # Deuterocanonical - Teal
    "TOB": "#16a085", "JDT": "#16a085", "1MA": "#16a085", "2MA": "#16a085",
    "WIS": "#16a085", "SIR": "#16a085", "BAR": "#16a085",
    # Wisdom/Poetry - Gold
    "JOB": "#f39c12", "PSA": "#f39c12", "PRO": "#f39c12", "ECC": "#f39c12", "SNG": "#f39c12",
    # Major Prophets - Red
    "ISA": "#e74c3c", "JER": "#e74c3c", "LAM": "#e74c3c", "EZK": "#e74c3c", "DAN": "#e74c3c",
    # Minor Prophets - Orange
    "HOS": "#e67e22", "JOL": "#e67e22", "AMO": "#e67e22", "OBA": "#e67e22", "JON": "#e67e22",
    "MIC": "#e67e22", "NAM": "#e67e22", "HAB": "#e67e22", "ZEP": "#e67e22", "HAG": "#e67e22",
    "ZEC": "#e67e22", "MAL": "#e67e22",
    # Gospels - Purple
    "MAT": "#9b59b6", "MRK": "#9b59b6", "LUK": "#9b59b6", "JHN": "#9b59b6",
    # Acts - Light Purple
    "ACT": "#8e44ad",
    # Pauline Epistles - Pink
    "ROM": "#e91e63", "1CO": "#e91e63", "2CO": "#e91e63", "GAL": "#e91e63", "EPH": "#e91e63",
    "PHP": "#e91e63", "COL": "#e91e63", "1TH": "#e91e63", "2TH": "#e91e63", "1TI": "#e91e63",
    "2TI": "#e91e63", "TIT": "#e91e63", "PHM": "#e91e63",
    # General Epistles - Cyan
    "HEB": "#00bcd4", "JAS": "#00bcd4", "1PE": "#00bcd4", "2PE": "#00bcd4",
    "1JN": "#00bcd4", "2JN": "#00bcd4", "3JN": "#00bcd4", "JUD": "#00bcd4",
    # Revelation - Dark Red
    "REV": "#c0392b",
}

def get_book_color(book_id):
    """Get color for a book, with fallback."""
    return BOOK_COLORS.get(book_id, "#95a5a6")

## Basic Statistics

Get an overview of the graph database contents.

In [4]:
with conn.session() as session:
    result = session.run("MATCH (v:Verse) RETURN count(v) AS count")
    print(f"Total verses: {result.single()['count']:,}")
    
    result = session.run("MATCH (v:Verse) RETURN count(DISTINCT v.book_id) AS count")
    print(f"Total books: {result.single()['count']}")
    
    result = session.run("MATCH ()-[r:CROSS_REFERENCES]->() RETURN count(r) AS count")
    print(f"Total cross-references: {result.single()['count']:,}")

Total verses: 35,817
Total books: 73
Total cross-references: 576,257


## Most Referenced Verses

Find the verses that are most frequently referenced by other verses.

In [5]:
with conn.session() as session:
    results = sq.run_query(session, sq.MOST_REFERENCED, limit=10)
    for r in results:
        print(f"{r['verse_id']} ({r['refs']} refs): {r['text'][:60]}...")

ISA-9-7 (183 refs): His reign will be increased, and there will be no end to his...
TIT-2-14 (177 refs): He gave himself for our sake, so that he might redeem us fro...
REV-5-9 (171 refs): And they were singing a new canticle, saying: “O Lord, you a...
1PE-2-9 (171 refs): But you are a chosen generation, a royal priesthood, a holy ...
ISA-9-6 (165 refs): For unto us a child is born, and unto us a son is given. And...
MAT-28-20 (162 refs): teaching them to observe all that I have ever commanded you....
REV-19-20 (159 refs): And the beast was apprehended, and with him the false prophe...
2CO-5-21 (147 refs): For God made him who did not know sin to be sin for us, so t...
TIT-3-5 (147 refs): And he saved us, not by works of justice that we had done, b...
ISA-55-7 (139 refs): Let the impious one abandon his way, and the iniquitous man ...


## Explore a Specific Verse

Look up a verse and see all its cross-references. Change `verse_id` to explore different verses.

In [6]:
verse_id = "GEN-1-1"

### Get the verse text

In [7]:
with conn.session() as session:
    result = session.run("MATCH (v:Verse {id: $id}) RETURN v.text AS text", id=verse_id)
    print(f"{verse_id}: {result.single()['text']}")

GEN-1-1: In the beginning, God created heaven and earth.


### Get cross-references from this verse

Shows verses that this verse references, sorted by vote confidence.

In [8]:
with conn.session() as session:
    results = sq.run_query(session, sq.VERSE_CROSSREFS, verse_id=verse_id)
    print(f"Cross-references from {verse_id}:")
    for r in results[:10]:
        print(f"  -> {r['target_id']} (votes: {r['votes']}): {r['text'][:50]}...")

Cross-references from GEN-1-1:
  -> JHN-1-1 (votes: 304): In the beginning was the Word, and the Word was wi...
  -> JHN-1-3 (votes: 304): All things were made through Him, and nothing that...
  -> JHN-1-2 (votes: 304): He was with God in the beginning....
  -> HEB-11-3 (votes: 238): By faith, we understand the world to be fashioned ...
  -> ISA-45-18 (votes: 200): For thus says the Lord, who created the heavens, G...
  -> REV-4-11 (votes: 165): “You are worthy, O Lord our God, to receive glory ...
  -> HEB-1-10 (votes: 158): And: “In the beginning, O Lord, you founded the ea...
  -> ISA-42-5 (votes: 134): Thus says the Lord God, who created the heavens an...
  -> COL-1-16 (votes: 133): For in him was created everything in heaven and on...
  -> COL-1-17 (votes: 133): And he is before all, and in him all things contin...


## Passage Groups

When TSK references a range of verses (e.g., Proverbs 8:22-30), those verses are grouped together with a shared `passage_group` UUID. This preserves the semantic meaning that these verses belong together as a unit.

In [9]:
with conn.session() as session:
    results = sq.run_query(session, sq.PASSAGE_GROUP_VERSES, source_id="GEN-1-1")
    for r in results[:5]:
        print(f"\nPassage Group: {r['passage_group'][:8]}...")
        print(f"Grouped verses: {r['grouped_verses']}")


Passage Group: 2970bf49...
Grouped verses: ['ROM-1-20', 'ROM-1-19']

Passage Group: afa1f9f9...
Grouped verses: ['COL-1-16', 'COL-1-17']

Passage Group: f500b38b...
Grouped verses: ['JHN-1-1', 'JHN-1-3', 'JHN-1-2']

Passage Group: fd7feebe...
Grouped verses: ['PSA-148-5', 'PSA-148-4']

Passage Group: 8ff5ef41...
Grouped verses: ['PRO-8-22', 'PRO-8-23', 'PRO-8-24', 'PRO-8-25', 'PRO-8-26', 'PRO-8-27', 'PRO-8-28', 'PRO-8-29', 'PRO-8-30']


## Path Finding

Find the shortest path between two verses through cross-references. This shows how any two verses in the Bible can be connected through chains of cross-references.

In [10]:
with conn.session() as session:
    results = sq.run_query(session, sq.SHORTEST_PATH, from_id="GEN-1-1", to_id="REV-22-21")
    if results:
        path = results[0]
        print(f"Path from Genesis to Revelation ({path['hops']} hops):")
        for verse in path['path']:
            print(f"  {verse}")
    else:
        print("No path found")

Path from Genesis to Revelation (3 hops):
  GEN-1-1
  GEN-2-4
  REV-1-4
  REV-22-21


## Book Interconnections

See which books of the Bible reference each other most frequently.

In [11]:
with conn.session() as session:
    results = sq.run_query(session, sq.BOOK_INTERCONNECTIONS, limit=15)
    print("Top book connections:")
    for r in results:
        print(f"  {r['from_book']} -> {r['to_book']}: {r['connections']:,} refs")

Top book connections:
  JER -> EZK: 3,767 refs
  JER -> ISA: 3,554 refs
  MAT -> LUK: 3,371 refs
  LUK -> MAT: 3,240 refs
  PSA -> ISA: 3,131 refs
  EZK -> JER: 3,002 refs
  ISA -> JER: 2,900 refs
  ISA -> PSA: 2,854 refs
  MRK -> MAT: 2,257 refs
  ISA -> EZK: 2,226 refs
  MAT -> MRK: 2,117 refs
  MAT -> JHN: 2,075 refs
  EZK -> ISA: 1,957 refs
  LUK -> JHN: 1,850 refs
  MRK -> LUK: 1,843 refs


## Graph Analytics (GDS)

Use Neo4j Graph Data Science plugin for advanced analytics like PageRank and community detection.

### Create graph projection

GDS algorithms operate on in-memory graph projections for performance.

In [12]:
with conn.session() as session:
    gds = GDSAnalytics(session)
    
    # Clean up any existing projection
    try:
        gds.drop_projection()
    except:
        pass
    
    proj = gds.create_projection()
    print(f"Created projection: {proj['graphName']}")
    print(f"  Nodes: {proj['nodeCount']:,}")
    print(f"  Relationships: {proj['relationshipCount']:,}")

Created projection: bible-graph
  Nodes: 35,817
  Relationships: 576,257


### PageRank

Find the most "important" verses based on the structure of cross-references. Verses referenced by many other important verses will rank higher.

In [13]:
with conn.session() as session:
    gds = GDSAnalytics(session)
    results = gds.pagerank(limit=10)
    print("Top verses by PageRank:")
    for r in results:
        print(f"  {r['verse_id']} (score: {r['pagerank']}): {r['text'][:50]}...")

Top verses by PageRank:
  TIT-2-14 (score: 8.4009): He gave himself for our sake, so that he might red...
  REV-5-9 (score: 8.0309): And they were singing a new canticle, saying: “O L...
  1PE-2-9 (score: 7.9049): But you are a chosen generation, a royal priesthoo...
  ISA-9-7 (score: 7.1445): His reign will be increased, and there will be no ...
  2CO-5-21 (score: 7.1064): For God made him who did not know sin to be sin fo...
  MAT-28-20 (score: 7.0905): teaching them to observe all that I have ever comm...
  TIT-3-5 (score: 7.0029): And he saved us, not by works of justice that we h...
  ISA-9-6 (score: 6.3515): For unto us a child is born, and unto us a son is ...
  REV-11-15 (score: 6.259): And the seventh Angel sounded the trumpet. And the...
  REV-19-20 (score: 6.2269): And the beast was apprehended, and with him the fa...


### Cleanup projection

Remove the in-memory projection when done to free resources.

In [14]:
with conn.session() as session:
    gds = GDSAnalytics(session)
    gds.drop_projection()
    print("Projection cleaned up")

Projection cleaned up


## Graph Visualization

Use pyvis to create interactive graph visualizations. Nodes are colored by book category:
- **Blue**: Pentateuch (Genesis-Deuteronomy)
- **Green**: Historical Books
- **Teal**: Deuterocanonical Books
- **Gold**: Wisdom/Poetry
- **Red**: Major Prophets
- **Orange**: Minor Prophets
- **Purple**: Gospels
- **Pink**: Pauline Epistles
- **Cyan**: General Epistles
- **Dark Red**: Revelation

### Visualize a verse and its connections

Create an interactive graph showing a verse and all verses it references or is referenced by.

In [15]:
def visualize_verse(verse_id, limit=50):
    """Create an interactive visualization of a verse and its connections.
    
    Args:
        verse_id: The verse to visualize (e.g., 'GEN-1-1')
        limit: Maximum number of connected nodes to show
    
    Returns:
        Network object. Call net.show('filename.html', notebook=True) to display.
    """
    net = Network(
        height="600px", 
        width="100%", 
        bgcolor="#222222", 
        font_color="white",
        notebook=True,
        cdn_resources='in_line'
    )
    net.barnes_hut(gravity=-3000, central_gravity=0.3, spring_length=200)
    
    query = """
    MATCH (center:Verse {id: $verse_id})
    OPTIONAL MATCH (center)-[r:CROSS_REFERENCES]-(connected:Verse)
    WITH center, connected, r
    LIMIT $limit
    RETURN center.id AS center_id, center.book_id AS center_book, 
           center.text AS center_text,
           connected.id AS conn_id, connected.book_id AS conn_book,
           connected.text AS conn_text,
           startNode(r).id AS from_id, endNode(r).id AS to_id,
           r.votes AS votes
    """
    
    added_nodes = set()
    
    with conn.session() as session:
        results = session.run(query, verse_id=verse_id, limit=limit)
        
        for record in results:
            # Add center node
            center_id = record["center_id"]
            if center_id not in added_nodes:
                net.add_node(
                    center_id,
                    label=center_id,
                    title=record["center_text"][:200],
                    color=get_book_color(record["center_book"]),
                    size=30,
                )
                added_nodes.add(center_id)
            
            # Add connected node
            conn_id = record["conn_id"]
            if conn_id and conn_id not in added_nodes:
                net.add_node(
                    conn_id,
                    label=conn_id,
                    title=record["conn_text"][:200] if record["conn_text"] else "",
                    color=get_book_color(record["conn_book"]),
                    size=20,
                )
                added_nodes.add(conn_id)
            
            # Add edge
            if record["from_id"] and record["to_id"]:
                votes = record["votes"] or 0
                net.add_edge(
                    record["from_id"],
                    record["to_id"],
                    title=f"votes: {votes}",
                    width=max(1, min(votes / 50, 5)),
                )
    
    print(f"Graph: {len(added_nodes)} nodes")
    return net

### Visualize Genesis 1:1

See all the verses connected to the first verse of the Bible.

In [16]:
net = visualize_verse("GEN-1-1", limit=30)
net.show("gen_1_1_graph.html")

Graph: 31 nodes
gen_1_1_graph.html


### Visualize a top PageRank verse

See the connections around one of the most "important" verses.

In [17]:
net = visualize_verse("ISA-9-6", limit=40)
net.show("isa_9_6_graph.html")

Graph: 41 nodes
isa_9_6_graph.html


### Visualize book-level connections

Create a high-level view showing how books of the Bible connect to each other.

In [21]:
def visualize_book_connections(min_connections=500):
    """Create a graph showing connections between books.
    
    Args:
        min_connections: Only show book pairs with at least this many cross-references
    
    Returns:
        Network object. Call net.show('filename.html', notebook=True) to display.
    """
    net = Network(
        height="700px", 
        width="100%", 
        bgcolor="#222222", 
        font_color="white",
        notebook=True,
        cdn_resources='in_line'
    )
    net.barnes_hut(gravity=-5000, central_gravity=0.5, spring_length=250)
    
    query = """
    MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
    WHERE a.book_id <> b.book_id
    WITH a.book_id AS from_book, b.book_id AS to_book, count(r) AS connections
    WHERE connections >= $min_connections
    RETURN from_book, to_book, connections
    ORDER BY connections DESC
    """
    
    added_nodes = set()
    
    with conn.session() as session:
        results = session.run(query, min_connections=min_connections)
        
        for record in results:
            from_book = record["from_book"]
            to_book = record["to_book"]
            connections = record["connections"]
            
            # Add book nodes
            for book in [from_book, to_book]:
                if book not in added_nodes:
                    net.add_node(
                        book,
                        label=book,
                        color=get_book_color(book),
                        size=25,
                    )
                    added_nodes.add(book)
            
            # Add edge
            net.add_edge(
                from_book,
                to_book,
                title=f"{connections:,} refs",
                width=max(1, connections / 500),
            )
    
    print(f"Graph: {len(added_nodes)} books")
    return net

In [19]:
net = visualize_book_connections(min_connections=1000)
net.show("book_connections_graph.html")

Graph: 21 books
book_connections_graph.html


## Cleanup

Close the database connection when done.

In [20]:
conn.close()
print("Connection closed")

Connection closed
